# Complete Pipeline: Merge & Quantize (Llama-3.2-3B)

**For models trained with `colab_simple.ipynb`**

## Step 1: Check What We Have

In [ ]:
# Check all directories
!ls -lh
print("\n" + "="*60)
!ls -lh out/ 2>/dev/null || echo "No out/ directory"
print("\n" + "="*60)
!ls -lh llama-3.2-3b-lora/ 2>/dev/null || echo "No llama-3.2-3b-lora/ directory"

## Step 2: Find LoRA Adapters

In [ ]:
import os

# Possible locations
possible_paths = [
    "./out",
    "./llama-3.2-3b-lora",
    "./llama-3.2-3b-finetuned",
]

# Also check for checkpoints
if os.path.exists("out"):
    checkpoints = [f"out/{d}" for d in os.listdir("out") if d.startswith("checkpoint-")]
    possible_paths.extend(checkpoints)

# Find the one with adapter files
adapter_path = None
for path in possible_paths:
    if os.path.exists(path):
        files = os.listdir(path)
        if "adapter_model.safetensors" in files or "adapter_model.bin" in files:
            adapter_path = path
            print(f"✅ Found LoRA adapters at: {adapter_path}")
            print(f"Files: {files}")
            break

if not adapter_path:
    print("❌ No LoRA adapters found!")
    print("\nSearched in:")
    for p in possible_paths:
        print(f"  - {p}")

## Step 3: Merge LoRA Adapters

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

if not adapter_path:
    print("❌ Cannot merge - no adapters found!")
else:
    print(f"Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        "meta-llama/Llama-3.2-3B-Instruct",
        torch_dtype=torch.float16,
        device_map="auto"
    )

    print(f"Loading LoRA adapters from {adapter_path}...")
    model = PeftModel.from_pretrained(base_model, adapter_path)

    print("Merging...")
    merged_model = model.merge_and_unload()

    print("Saving merged model...")
    merged_model.save_pretrained("./merged-llama-3.2-3b")

    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
    tokenizer.save_pretrained("./merged-llama-3.2-3b")

    print("✅ Merged model saved to ./merged-llama-3.2-3b")

    # Clean up
    del base_model, model, merged_model
    torch.cuda.empty_cache()

## Step 4: Install AWQ

In [ ]:
!pip install -q autoawq

## Step 5: Quantize with AWQ

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

print("Loading merged model...")
model = AutoAWQForCausalLM.from_pretrained(
    "./merged-llama-3.2-3b",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("./merged-llama-3.2-3b")

quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

print("Quantizing... (~20-30 min)")
model.quantize(tokenizer, quant_config=quant_config)

print("Saving...")
model.save_quantized("./quantized-llama-3.2-3b-awq")
tokenizer.save_pretrained("./quantized-llama-3.2-3b-awq")

print("✅ Quantized model saved!")

## Step 6: Test

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model = AutoAWQForCausalLM.from_quantized("./quantized-llama-3.2-3b-awq", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("./quantized-llama-3.2-3b-awq")

def chat(prompt):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Test 1: Summarization")
print(chat("Summarize: AI is transforming the world."))

print("\nTest 2: Chat")
print(chat("Hello! How are you?"))

print("\nTest 3: Persona Response")
print(chat("Tell me a joke"))

## Step 7: Download

In [ ]:
!zip -r quantized-llama-3.2-3b-awq.zip quantized-llama-3.2-3b-awq/
print("✅ Download: quantized-llama-3.2-3b-awq.zip")